In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
def calculate_auc_ci(y_true, y_scores, confidence_level=0.95):
    fpr_values, tpr_values, _ = roc_curve(y_true, y_scores)
    auc_value = auc(fpr_values, tpr_values)
    
    n_bootstraps = 1000
    rng = np.random.RandomState(42)
    bootstrapped_scores = []
    
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_scores), len(y_scores))
        if len(np.unique(y_true[indices])) < 2:
            continue
        fpr_bootstrap, tpr_bootstrap, _ = roc_curve(y_true[indices], y_scores[indices])
        score = auc(fpr_bootstrap, tpr_bootstrap)
        bootstrapped_scores.append(score)
        
    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    confidence_lower = sorted_scores[int((1 - confidence_level) / 2 * len(sorted_scores))]
    confidence_upper = sorted_scores[int((1 + confidence_level) / 2 * len(sorted_scores))]
    
    return auc_value, confidence_lower, confidence_upper

In [ ]:
train_path = 'path/train_PCA.xlsx'
df = pd.read_excel(train_path,sheet_name='Sheet1')
X = df.iloc[:, 1:]
y = df.iloc[:,0]
X, y = np.array(X), np.array(y)

In [ ]:
save_dir = 'path/'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
param_test = {
    'n_estimators': [100,200,300,500],
    'max_depth': [3,4,5],
    'learning_rate': [0.01, 0.1],
    'min_child_weight': [3, 5],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0.1, 0.3],
    'reg_alpha': [0.1, 0.5],
    'reg_lambda': [1, 2, 3]
}

xgb = XGBClassifier(n_jobs=-1, tree_method='hist', random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_res = GridSearchCV(estimator=xgb, 
                       param_grid=param_test, 
                       cv=cv, 
                       scoring='roc_auc', 
                       return_train_score=True)

xgb_res.fit(X, y)
print("Best Parameters:", xgb_res.best_params_)
print("Best ROC AUC Score:", xgb_res.best_score_)
pd.DataFrame(xgb_res.cv_results_).to_excel(os.path.join(save_dir, 'grid_search_results.xlsx'), index=False)

In [ ]:
estimator = XGBClassifier(**xgb_res.best_params_)
estimator.fit(X, y)
y_proba_train = estimator.predict_proba(X)[:,1]
fpr, tpr, thresholds = metrics.roc_curve(y, y_proba_train)
roc_auc = auc(fpr, tpr)
j_scores = tpr + (1 - fpr) - 1
max_j_index = np.argmax(j_scores)
best_threshold = thresholds[max_j_index]
y_pred_train =(y_proba_train >= best_threshold).astype(int)

results_df = pd.DataFrame({'Actual': y, 'Predicted': y_pred_train, 'Probability': y_proba_train})
results_df.to_excel(os.path.join(save_dir,'prediction_train.xlsx'), index=False)  

accuracy = metrics.accuracy_score(y, y_pred_train)
precision, recall, fscore, _ = metrics.precision_recall_fscore_support(y, y_pred_train, average='binary')
specificity = metrics.recall_score(y, y_pred_train, pos_label=1)
results_dict = {'AUC':[roc_auc],'Accuracy': [accuracy],'Precision': [precision],'Recall': [recall],'F1 Score': [fscore],
                'Specificity': [specificity]}
results_df = pd.DataFrame(results_dict)
results_df.to_excel(os.path.join(save_dir,'train_metrics_results.xlsx'), index=False)

best_model = estimator
joblib.dump(best_model, os.path.join(save_dir, 'best_model.pkl'))

In [ ]:
auc_value, auc_ci_lower, auc_ci_upper = calculate_auc_ci(y, y_proba_train)

fpr, tpr, _ = roc_curve(y, y_proba_train)
plt.figure(facecolor='white', figsize=(4, 4))
plt.plot(fpr, tpr, color='#FFBE7A', alpha=1, 
         label=f"AUC = {auc_value:.3f} \n(95% CI [{auc_ci_lower:.3f}, {auc_ci_upper:.3f}])", linewidth=1.5)
plt.fill_between(fpr, tpr, color='#FFBE7A', alpha=0.1) 
plt.scatter(fpr[max_j_index], tpr[max_j_index], color='#FFBE7A', marker='*', s=100)
plt.xlabel('False Positive Rate(1-Specificity)', fontsize=12)
plt.ylabel('True Positive Rate(Sensitivity)', fontsize=12)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.legend(fontsize=10, loc='lower right', frameon=False)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax = plt.gca()
ax.spines['top'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.tick_params(axis='both', width=1) 
plt.savefig(os.path.join(save_dir, 'ROC_train.pdf'), dpi=300)
plt.show()